In [ ]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, 'w') as f:
            f.write(text_data)
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

file_path = "datasets/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print(len(data))
            

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
    )
    return instruction_text + input_text

model_input = format_input(data[999])
desired_output = data[999]['output']
print(model_input + desired_output)

In [ ]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
valid_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
valid_data = data[train_portion + test_portion:]
print(f"Train data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")
print(f"Validation data size: {len(valid_data)}")

In [ ]:
import torch
from torch.utils.data import Dataset
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.encoded_texts[idx]

## 自定义批处理聚合函数

In [ ]:
def custom_collate_fn(
        batch,
        pad_token_id=50256,
        ignore_index=-100,
        allowed_max_length=None,
        device="cpu"
):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst = []
    target_lst = []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            targets = targets[:allowed_max_length]
            inputs = inputs[:allowed_max_length]
        inputs_lst.append(inputs)
        target_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, targets_tensor

In [ ]:
input1 = [0, 1, 2, 3, 4]
input2 = [5, 6, 7]
input3 = [9, 10]

batch = (
    input1,
    input2,
    input3
)
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

## 探索为什么-100

In [ ]:
logit1 = torch.tensor([[-1.0, 1.0],
                       [-0.5, 1.5],
                       [-0.5, 1.5]])
target1 = torch.tensor([0, 1, -100])
loss1 = torch.nn.functional.cross_entropy(logit1, target1)
print(loss1)

## 创建数据loader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from functools import partial
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)


In [ ]:
from torch.utils.data import DataLoader
import tiktoken
num_workers = 0
batch_size = 8
tokenizer = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer=tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
valid_dataset = InstructionDataset(valid_data, tokenizer=tokenizer)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
test_dataset = InstructionDataset(test_data, tokenizer=tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)


In [ ]:
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)
    break

## 加载预训练模型

In [ ]:
from gpt_download import download_and_load_gpt2
from models.GPTModel import GPTModel
from utils import load_weights_into_gpt, generate, text_to_token_ids, token_ids_to_text

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "dropout": 0.1,
    "bias": True,
}
model_config = {
    "gpt2-small" : {"emb_dim":768, "num_layers":12, "num_heads":12},
    "gpt2-medium" : {"emb_dim":1024, "num_layers":24, "num_heads":16},
    "gpt2-large" : {"emb_dim":1280, "num_layers":36, "num_heads":20},
    "gpt2-xl" : {"emb_dim":1600, "num_layers":48, "num_heads":25}
}

BASE_CONFIG.update(model_config["gpt2-medium"])
settings, params = download_and_load_gpt2(
    model_size="355M",
    models_dir="../models/"
)
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
torch.manual_seed(123)
input_text = format_input(valid_data[0])
print(input_text)

In [ ]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
response_test = generated_text[len(input_text):].strip()
print("Generated response:\n", response_test)

## 微调大模型

In [18]:
from chapter5.train import train_model_simple
from chapter5.calc_loss import calc_loss_loader

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, eval_iter)
        valid_loss = calc_loss_loader(val_loader, model, device, eval_iter)
    model.train()
    return train_loss, valid_loss



In [ ]:
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    valid_loss = calc_loss_loader(valid_loader, model, device, num_batches=5)
print(f"Initial Train Loss: {train_loss:.4f}, Initial Valid Loss: {valid_loss:.4f}")

In [ ]:
import time
start_time = time.time()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-1)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, valid_loader, optimizer, device,
    epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(valid_data[0]),
    tokenizer=tokenizer
)
end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

## 测试并保留测试数据集上的答复

In [19]:
for entry in test_data[:3]:
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_test = generated_text[len(input_text):].replace("### Response:", "").strip()
    print("输入内容:\n", input_text)
    print("生成答复:\n", response_test)
    print("预期答复:\n", entry['output'])
    print("-" * 80)

输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.
生成答复:
 The car is as fast as a cheetah.
预期答复:
 The car is as fast as lightning.
--------------------------------------------------------------------------------
输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?
生成答复:
 A thunderstorm is a type of thunderstorm.
预期答复:
 The type of cloud typically associated with thunderstorms is cumulonimbus.
--------------------------------------------------------------------------------
输入内容:
 Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.
生成答复:
 The author of 'Pride and Prejudice' is George Bernard 

In [ ]:
from tqdm import tqdm
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256,
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_test = generated_text[len(input_text):].replace("### Response:", "").strip()
    test_data[i]['generated_response'] = response_test
with open("datasets/instruction-data-with-responses.json", "w") as f:
    json.dump(test_data, f, indent=4)

In [ ]:
print(test_data[0])

In [ ]:
import psutil
def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")
print("Is ollama running?", ollama_running)

In [ ]:
import urllib.request

def query_model(
        prompt,
        model_name="qwen3:4b",
        url="http://localhost:11434/api/chat",
): 
    data = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {
            "seed": 123,
            "temperature": 0.0,
            "num_ctx": 2048
        }
    }
    payload = json.dumps(data).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=payload,
        headers={
            "Content-Type": "application/json"
        },
        method="POST"
    )
    response_data = ""
    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")
            if not line: 
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]
    return response_data

In [ ]:
import json
model = "gemma3:4b"
result = query_model("what do Llamas eat?", model_name=model)
print(result)

In [ ]:
from tqdm import tqdm
def generate_model_scores(json_data, json_key, model):
    scores = []
    for entry in tqdm(json_data, desc='Generating model scores'):
        prompt = (
            f"Given the input '{format_input(entry)}', "
            f"and correct output '{entry['output']}', "
            f"score the model response '{entry[json_key]}'"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with only the integer score."
        )
        score = query_model(prompt, model)
        scores.append(score)
    return scores

In [ ]:
with open('datasets/instruction-data-with-responses.json', 'r', encoding='utf-8') as file:
    # 读取并解析文件
    test_data = json.load(file)
scores = generate_model_scores(test_data, "generated_response", "gemma3:4b")

In [ ]:
res = 0
for score in scores:
    res += int(score.strip('\n'))
print(res / len(scores))